# Notebook 02 — Data Cleaning and Preparation

## 1. Import libraries and configure paths

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)


Pandas version: 3.1.0
NumPy version: 2.4.6


In [ ]:
cwd = Path.cwd().resolve()

if (cwd / "data" / "raw").is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "raw").is_dir() and cwd.name.lower() == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    
    candidates = [cwd, *cwd.parents]
    PROJECT_ROOT = next(
        (p for p in candidates if (p / "data" / "raw").is_dir()),
        None
    )

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project data/raw folder. Open this notebook "
        "from the project root or notebooks/ folder, and confirm Notebook 01 "
        "has saved the raw CSV under data/raw/."
    )

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)


Project root: D:\global seismic trends
Raw data directory: D:\global seismic trends\data\raw
Processed data directory: D:\global seismic trends\data\processed


## 2. Find and load the latest raw CSV

In [3]:
# Notebook 01 writes timestamped USGS earthquake CSV files.
# Exclude any other CSV exports that are not the earthquake data itself.
raw_candidates = [
    p for p in RAW_DIR.glob("usgs_earthquakes_*.csv")
    if p.is_file()
]

if not raw_candidates:
    raise FileNotFoundError(
        f"No file matching 'usgs_earthquakes_*.csv' was found in {RAW_DIR}. "
        "Run Notebook 01 first and verify that the raw CSV was saved."
    )

raw_file = max(raw_candidates, key=lambda p: p.stat().st_mtime)
print("Selected raw file:", raw_file.name)
print("File size (MB):", round(raw_file.stat().st_size / (1024 ** 2), 2))

df_raw = pd.read_csv(raw_file, low_memory=False)
print("Raw dataset shape:", df_raw.shape)
display(df_raw.head())


Selected raw file: usgs_earthquakes_20261022T132512Z.csv
File size (MB): 280.48
Raw dataset shape: (739554, 34)


,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,tsunami,sig,net,nst,dmin,rms,gap,magError,depthError,magNst,locationSource,magSource,types,ids,sources,type,alert,felt,cdi,mmi,url,detail,place_type,time_zone
0,nc73642735,2021-10-22 13:22:28.170+00:00,2021-10-22 15:26:11.754+00:00,38.807835,-122.813835,0.08,0.84,md,"6km NW of The Geysers, CA",automatic,0,11,nc,11.0,0.01203,0.02,77.0,NaN,NaN,NaN,NaN,NaN,",nearby-cities,origin,phase-data,scitech-link,",",nc73642735,",",nc,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventpage/nc73642735,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=nc73642735&format=geojson,NaN,NaN
1,mb80527769,2021-10-22 13:27:15.120+00:00,2021-10-22 15:21:51.820+00:00,47.839167,-114.289000,3.17,-0.20,ml,"3 km SSW of Dayton, Montana",reviewed,0,0,mb,6.0,0.09300,0.04,169.0,NaN,NaN,NaN,NaN,NaN,",origin,phase-data,",",mb80527769,",",mb,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventpage/mb80527769,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=mb80527769&format=geojson,NaN,NaN
2,mb80527774,2021-10-22 13:31:59.150+00:00,2021-10-22 15:28:57.720+00:00,47.766833,-113.753667,3.91,0.56,ml,"19 km SSE of Swan Lake, Montana",reviewed,0,5,mb,7.0,0.19900,0.11,148.0,NaN,NaN,NaN,NaN,NaN,",origin,phase-data,",",mb80527774,",",mb,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventpage/mb80527774,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=mb80527774&format=geojson,NaN,NaN
3,ak021dk3jgyc,2021-10-22 13:44:11.726+00:00,2021-12-06 18:54:05.882+00:00,59.678800,-136.572300,24.00,1.20,ml,"31 km WNW of Mosquito Lake, Alaska",reviewed,0,22,ak,NaN,NaN,0.26,NaN,NaN,NaN,NaN,NaN,NaN,",origin,phase-data,",",ak021dk3jgyc,",",ak,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventpage/ak021dk3jgyc,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=ak021dk3jgyc&format=geojson,NaN,NaN
4,ci39840223,2021-10-22 13:55:21.010+00:00,2021-10-22 17:59:28.696+00:00,33.344000,-116.928500,18.04,0.79,ml,"6km WSW of Palomar Observatory, CA",reviewed,0,10,ci,25.0,0.05595,0.15,48.0,NaN,NaN,NaN,NaN,NaN,",nearby-cities,origin,phase-data,scitech-link,",",ci39840223,",",ci,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventpage/ci39840223,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=ci39840223&format=geojson,NaN,NaN


## 3. Check expected columns

In [4]:
CORE_COLUMNS = [
    "id", "time", "updated", "latitude", "longitude", "depth_km", "mag",
    "magType", "place", "status", "tsunami", "sig", "net", "nst", "dmin",
    "rms", "gap", "magError", "depthError", "magNst", "locationSource",
    "magSource", "types", "ids", "sources", "type"
]

present_core_columns = [c for c in CORE_COLUMNS if c in df_raw.columns]
missing_core_columns = [c for c in CORE_COLUMNS if c not in df_raw.columns]

print(f"Core columns present: {len(present_core_columns)} / {len(CORE_COLUMNS)}")
print("Missing expected core columns:", missing_core_columns if missing_core_columns else "None")
print("Additional columns:", [c for c in df_raw.columns if c not in CORE_COLUMNS])

if "id" not in df_raw.columns:
    raise KeyError("Required field 'id' is missing from the raw CSV.")
if "time" not in df_raw.columns:
    raise KeyError("Required field 'time' is missing from the raw CSV.")


Core columns present: 26 / 26
Missing expected core columns: None
Additional columns: ['alert', 'felt', 'cdi', 'mmi', 'url', 'detail', 'place_type', 'time_zone']


## 4. Create a working copy and standardize column names

In [ ]:
df = df_raw.copy()

# Strip whitespace from column labels
df.columns = [str(c).strip() for c in df.columns]

text_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()
for col in text_columns:
    df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)
    df[col] = df[col].replace({"": pd.NA})

print("Working copy shape:", df.shape)


Working copy shape: (739554, 34)


## 5. Standardize timestamps and numeric fields

In [ ]:
# Parse timestamps consistently as timezone-aware UTC datetimes.
for col in ["time", "updated"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")

NUMERIC_COLUMNS = [
    "latitude", "longitude", "depth_km", "mag", "tsunami", "sig", "nst",
    "dmin", "rms", "gap", "magError", "depthError", "magNst"
]
numeric_columns_present = [c for c in NUMERIC_COLUMNS if c in df.columns]

for col in numeric_columns_present:
    df[col] = pd.to_numeric(df[col], errors="coerce")

if "tsunami" in df.columns:
    df["tsunami"] = df["tsunami"].round().astype("Int64")

print("Timestamp conversion complete.")
print("Numeric fields converted:", numeric_columns_present)
print("Dtypes:")
display(df.dtypes.rename("dtype").to_frame())


Timestamp conversion complete.
Numeric fields converted: ['latitude', 'longitude', 'depth_km', 'mag', 'tsunami', 'sig', 'nst', 'dmin', 'rms', 'gap', 'magError', 'depthError', 'magNst']
Dtypes:


,dtype
id,str
time,"datetime64[us, UTC]"
updated,"datetime64[us, UTC]"
latitude,float64
longitude,float64
depth_km,float64
mag,float64
magType,str
place,str
status,str


## 6. Inspect missing values 

In [7]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "dtype": df.dtypes.astype(str)
}).sort_values(["missing_percent", "missing_count"], ascending=False)

display(missing_summary)

print("Total rows:", len(df))
print("Rows missing event ID:", int(df["id"].isna().sum()))
print("Rows missing parsed timestamp:", int(df["time"].isna().sum()))

fully_missing_columns = [c for c in df.columns if df[c].isna().all()]
print("Columns that are completely missing:", fully_missing_columns if fully_missing_columns else "None")


,missing_count,missing_percent,dtype
magError,739554,100.00,float64
depthError,739554,100.00,float64
magNst,739554,100.00,float64
locationSource,739554,100.00,float64
magSource,739554,100.00,float64
place_type,739554,100.00,float64
time_zone,739554,100.00,float64
alert,735465,99.45,str
mmi,729991,98.71,float64
felt,702699,95.02,float64


Total rows: 739554
Rows missing event ID: 0
Rows missing parsed timestamp: 0
Columns that are completely missing: ['magError', 'depthError', 'magNst', 'locationSource', 'magSource', 'place_type', 'time_zone']


## 7. Check identifiers and duplicate events

In [ ]:
rows_before_dedup = len(df)
missing_id_rows = int(df["id"].isna().sum())
duplicate_id_rows = int(df["id"].duplicated(keep="first").sum())

non_missing_id = df["id"].notna()
duplicate_non_missing_id = non_missing_id & df["id"].duplicated(keep="first")
df = df.loc[~duplicate_non_missing_id].copy()

print("Rows before deduplication:", rows_before_dedup)
print("Duplicate non-missing IDs removed:", int(duplicate_non_missing_id.sum()))
print("Rows with missing ID retained for review:", missing_id_rows)
print("Rows after deduplication:", len(df))
print("Remaining duplicate non-missing IDs:",
      int(df.loc[df["id"].notna(), "id"].duplicated().sum()))


Rows before deduplication: 739554
Duplicate non-missing IDs removed: 0
Rows with missing ID retained for review: 0
Rows after deduplication: 739554
Remaining duplicate non-missing IDs: 0


## 8. Validate coordinate and measurement ranges

In [9]:
# Quality flags are retained in the dataset; they do not automatically remove rows.
if {"latitude", "longitude"}.issubset(df.columns):
    df["is_invalid_latitude"] = df["latitude"].notna() & ~df["latitude"].between(-90, 90)
    df["is_invalid_longitude"] = df["longitude"].notna() & ~df["longitude"].between(-180, 180)
    df["has_invalid_coordinates"] = df["is_invalid_latitude"] | df["is_invalid_longitude"]
else:
    df["is_invalid_latitude"] = pd.NA
    df["is_invalid_longitude"] = pd.NA
    df["has_invalid_coordinates"] = pd.NA

if "depth_km" in df.columns:
    df["is_negative_depth"] = df["depth_km"].notna() & (df["depth_km"] < 0)
    df["is_extreme_depth_over_700km"] = df["depth_km"].notna() & (df["depth_km"] > 700)
else:
    df["is_negative_depth"] = pd.NA
    df["is_extreme_depth_over_700km"] = pd.NA

if "mag" in df.columns:
    df["is_magnitude_outside_review_range"] = df["mag"].notna() & ~df["mag"].between(-2, 10)
    df["has_missing_magnitude"] = df["mag"].isna()
else:
    df["is_magnitude_outside_review_range"] = pd.NA
    df["has_missing_magnitude"] = pd.NA

if "time" in df.columns:
    df["has_missing_timestamp"] = df["time"].isna()
else:
    df["has_missing_timestamp"] = pd.NA

quality_flag_summary = pd.DataFrame({
    "flag": [
        "is_invalid_latitude",
        "is_invalid_longitude",
        "has_invalid_coordinates",
        "is_negative_depth",
        "is_extreme_depth_over_700km",
        "is_magnitude_outside_review_range",
        "has_missing_magnitude",
        "has_missing_timestamp"
    ],
    "flagged_rows": [
        int(df[c].fillna(False).sum()) if c in df.columns else 0
        for c in [
            "is_invalid_latitude", "is_invalid_longitude", "has_invalid_coordinates",
            "is_negative_depth", "is_extreme_depth_over_700km",
            "is_magnitude_outside_review_range", "has_missing_magnitude",
            "has_missing_timestamp"
        ]
    ]
})
display(quality_flag_summary)

print("Important: flagged rows are retained for investigation unless a later, documented analysis rule excludes them.")


,flag,flagged_rows
0,is_invalid_latitude,0
1,is_invalid_longitude,0
2,has_invalid_coordinates,0
3,is_negative_depth,25047
4,is_extreme_depth_over_700km,0
5,is_magnitude_outside_review_range,75
6,has_missing_magnitude,57
7,has_missing_timestamp,0


Important: flagged rows are retained for investigation unless a later, documented analysis rule excludes them.


## 9. Derive time-based features

In [10]:
if "time" in df.columns:
    df["year"] = df["time"].dt.year.astype("Int64")
    df["month"] = df["time"].dt.month.astype("Int64")
    df["day"] = df["time"].dt.day.astype("Int64")
    df["day_of_week"] = df["time"].dt.day_name()
    df["hour_utc"] = df["time"].dt.hour.astype("Int64")
    df["date_utc"] = df["time"].dt.strftime("%Y-%m-%d")
    df.loc[df["time"].isna(), "date_utc"] = pd.NA

display(df[[c for c in ["time", "date_utc", "year", "month", "day", "day_of_week", "hour_utc"] if c in df.columns]].head())


,time,date_utc,year,month,day,day_of_week,hour_utc
0,2021-10-22 13:22:28.170+00:00,2021-10-22,2021,10,22,Friday,13
1,2021-10-22 13:27:15.120+00:00,2021-10-22,2021,10,22,Friday,13
2,2021-10-22 13:31:59.150+00:00,2021-10-22,2021,10,22,Friday,13
3,2021-10-22 13:44:11.726+00:00,2021-10-22,2021,10,22,Friday,13
4,2021-10-22 13:55:21.010+00:00,2021-10-22,2021,10,22,Friday,13


## 10. Create depth and magnitude categories

In [11]:
if "depth_km" in df.columns:
    depth_conditions = [
        df["depth_km"] < 0,
        df["depth_km"].between(0, 69.999999, inclusive="both"),
        df["depth_km"].between(70, 300, inclusive="both"),
        df["depth_km"] > 300
    ]
    depth_labels = [
        "Review - negative",
        "Shallow (<70 km)",
        "Intermediate (70-300 km)",
        "Deep (>300 km)"
    ]
    df["depth_category"] = np.select(
        depth_conditions,
        depth_labels,
        default="Unknown"
    )
    df.loc[df["depth_km"].isna(), "depth_category"] = pd.NA

if "mag" in df.columns:
    mag_bins = [-np.inf, 2, 3, 4, 5, 6, 7, np.inf]
    mag_labels = [
        "Below 2.0", "2.0-2.9", "3.0-3.9", "4.0-4.9",
        "5.0-5.9", "6.0-6.9", "7.0+"
    ]
    df["magnitude_category"] = pd.cut(
        df["mag"],
        bins=mag_bins,
        labels=mag_labels,
        right=False
    ).astype("string")
    df.loc[df["mag"].isna(), "magnitude_category"] = pd.NA
    if "is_magnitude_outside_review_range" in df.columns:
        df.loc[df["is_magnitude_outside_review_range"].fillna(False),
               "magnitude_category"] = "Review - unusual magnitude"

display(df[[c for c in ["depth_km", "depth_category", "mag", "magnitude_category"] if c in df.columns]].head(10))


,depth_km,depth_category,mag,magnitude_category
0,0.08,Shallow (<70 km),0.84,Below 2.0
1,3.17,Shallow (<70 km),-0.20,Below 2.0
2,3.91,Shallow (<70 km),0.56,Below 2.0
3,24.00,Shallow (<70 km),1.20,Below 2.0
4,18.04,Shallow (<70 km),0.79,Below 2.0
5,2.70,Shallow (<70 km),2.14,2.0-2.9
6,10.27,Shallow (<70 km),0.82,Below 2.0
7,3.37,Shallow (<70 km),1.17,Below 2.0
8,3.71,Shallow (<70 km),1.10,Below 2.0
9,16.00,Shallow (<70 km),3.00,3.0-3.9


## 11. Review place text

In [12]:
if "place" in df.columns:
    df["place_region_text"] = (
        df["place"]
        .astype("string")
        .str.replace(r"^\s*\d+(?:\.\d+)?\s*km\s+", "", regex=True)
        .str.strip()
    )
    df.loc[df["place"].isna(), "place_region_text"] = pd.NA

display(df[[c for c in ["place", "place_region_text"] if c in df.columns]].head(10))


,place,place_region_text
0,"6km NW of The Geysers, CA","NW of The Geysers, CA"
1,"3 km SSW of Dayton, Montana","SSW of Dayton, Montana"
2,"19 km SSE of Swan Lake, Montana","SSE of Swan Lake, Montana"
3,"31 km WNW of Mosquito Lake, Alaska","WNW of Mosquito Lake, Alaska"
4,"6km WSW of Palomar Observatory, CA","WSW of Palomar Observatory, CA"
5,"7km ENE of Walker, CA","ENE of Walker, CA"
6,"11km NE of San Martin, CA","NE of San Martin, CA"
7,"7km E of Walker, CA","E of Walker, CA"
8,"7km E of Walker, CA","E of Walker, CA"
9,"110 km SSE of Sand Point, Alaska","SSE of Sand Point, Alaska"


## 12. Final data-quality review

In [13]:
print("Cleaned dataset shape:", df.shape)
print("Duplicate non-missing IDs remaining:",
      int(df.loc[df["id"].notna(), "id"].duplicated().sum()))
print("Missing IDs:", int(df["id"].isna().sum()))
print("Missing timestamps:", int(df["time"].isna().sum()) if "time" in df.columns else "time column absent")

print("\nKey numeric summary:")
summary_cols = [c for c in ["mag", "depth_km", "latitude", "longitude", "sig", "nst", "gap", "rms"] if c in df.columns]
display(df[summary_cols].describe().T if summary_cols else pd.DataFrame())

print("\nDerived category counts:")
for col in ["depth_category", "magnitude_category"]:
    if col in df.columns:
        display(df[col].value_counts(dropna=False).rename("count").to_frame())


Cleaned dataset shape: (739554, 51)
Duplicate non-missing IDs remaining: 0
Missing IDs: 0
Missing timestamps: 0

Key numeric summary:


,count,mean,std,min,25%,50%,75%,max
mag,739497.0,1.657911,1.298119,-5.0000,0.850000,1.400000,2.100000,8.8000
depth_km,739554.0,24.519012,55.651065,-10.0000,3.480000,8.355700,18.100000,683.5780
latitude,739554.0,39.916857,19.973895,-84.4932,33.534333,38.824333,55.874075,87.0815
longitude,739554.0,-110.117216,75.775555,-179.9997,-152.254833,-122.765667,-112.906333,179.9994
sig,739554.0,68.361315,98.658151,0.0000,11.000000,30.000000,68.000000,2910.0000
nst,580689.0,24.316615,23.559887,0.0000,10.000000,18.000000,30.000000,619.0000
gap,591508.0,116.617205,65.227004,8.0000,68.000000,99.000000,152.000000,360.0000
rms,739497.0,0.312451,0.282026,0.0000,0.100000,0.200000,0.500000,8.5500



Derived category counts:


,count
depth_category,
Shallow (<70 km),642745
Intermediate (70-300 km),65744
Review - negative,25047
Deep (>300 km),6018


,count
magnitude_category,
Below 2.0,525467
2.0-2.9,110351
4.0-4.9,71356
3.0-3.9,23138
5.0-5.9,8448
6.0-6.9,591
Review - unusual magnitude,75
7.0+,71
<NA>,57


## 13. Save the cleaned dataset and cleaning metadata

In [ ]:
run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
cleaned_csv = PROCESSED_DIR / f"usgs_earthquakes_cleaned_{run_timestamp}.csv"
metadata_json = PROCESSED_DIR / f"usgs_earthquakes_cleaning_metadata_{run_timestamp}.json"

df.to_csv(cleaned_csv, index=False, encoding="utf-8")

metadata = {
    "project": "Global Seismic Trends: Data-Driven Earthquake Insights",
    "notebook": "02_data_cleaning.ipynb",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_file": str(raw_file),
    "output_file": str(cleaned_csv),
    "rows_before_cleaning": int(len(df_raw)),
    "rows_after_deduplication_and_cleaning": int(len(df)),
    "columns_in_output": int(len(df.columns)),
    "duplicate_non_missing_ids_removed": int(duplicate_non_missing_id.sum()),
    "missing_id_rows_retained": int(df["id"].isna().sum()),
    "missing_timestamp_rows": int(df["time"].isna().sum()) if "time" in df.columns else None,
    "fully_missing_columns": [c for c in df.columns if df[c].isna().all()],
    "missing_value_policy": "No blanket imputation; missing measurements remain null.",
    "negative_depth_policy": "Flagged for review and retained.",
    "coordinate_policy": "Invalid coordinates flagged; rows retained.",
    "country_continent_policy": "Not inferred from place text; geospatial enrichment deferred.",
    "derived_columns": [
        c for c in [
            "year", "month", "day", "day_of_week", "hour_utc", "date_utc",
            "depth_category", "magnitude_category", "place_region_text"
        ] if c in df.columns
    ]
}

with open(metadata_json, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False, default=str)

print("Cleaned CSV saved to:", cleaned_csv)
print("Metadata saved to:", metadata_json)
print("CSV size (MB):", round(cleaned_csv.stat().st_size / (1024 ** 2), 2))


Cleaned CSV saved to: D:\global seismic trends\data\processed\usgs_earthquakes_cleaned_20261022T134538Z.csv
Metadata saved to: D:\global seismic trends\data\processed\usgs_earthquakes_cleaning_metadata_20261022T134538Z.json
CSV size (MB): 373.63


## 14. Reload check

In [15]:
df_saved_check = pd.read_csv(cleaned_csv, low_memory=False)
assert len(df_saved_check) == len(df), "Saved CSV row count does not match the cleaned dataframe."
assert "id" in df_saved_check.columns, "Saved output is missing the event ID column."
assert "time" in df_saved_check.columns, "Saved output is missing the event timestamp column."

print("Reload validation passed.")
print("Saved shape:", df_saved_check.shape)
display(df_saved_check.head(5))


Reload validation passed.
Saved shape: (739554, 51)


,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,tsunami,sig,net,nst,dmin,rms,gap,magError,depthError,magNst,locationSource,magSource,types,ids,sources,type,alert,felt,cdi,mmi,url,detail,place_type,time_zone,is_invalid_latitude,is_invalid_longitude,has_invalid_coordinates,is_negative_depth,is_extreme_depth_over_700km,is_magnitude_outside_review_range,has_missing_magnitude,has_missing_timestamp,year,month,day,day_of_week,hour_utc,date_utc,depth_category,magnitude_category,place_region_text
0,nc73642735,2021-10-22 13:22:28.170+00:00,2021-10-22 15:26:11.754+00:00,38.807835,-122.813835,0.08,0.84,md,"6km NW of The Geysers, CA",automatic,0,11,nc,11.0,0.01203,0.02,77.0,NaN,NaN,NaN,NaN,NaN,",nearby-cities,origin,phase-data,scitech-link,",",nc73642735,",",nc,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventpage/nc73642735,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=nc73642735&format=geojson,NaN,NaN,False,False,False,False,False,False,False,False,2021,10,22,Friday,13,2021-10-22,Shallow (<70 km),Below 2.0,"NW of The Geysers, CA"
1,mb80527769,2021-10-22 13:27:15.120+00:00,2021-10-22 15:21:51.820+00:00,47.839167,-114.289000,3.17,-0.20,ml,"3 km SSW of Dayton, Montana",reviewed,0,0,mb,6.0,0.09300,0.04,169.0,NaN,NaN,NaN,NaN,NaN,",origin,phase-data,",",mb80527769,",",mb,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventpage/mb80527769,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=mb80527769&format=geojson,NaN,NaN,False,False,False,False,False,False,False,False,2021,10,22,Friday,13,2021-10-22,Shallow (<70 km),Below 2.0,"SSW of Dayton, Montana"
2,mb80527774,2021-10-22 13:31:59.150+00:00,2021-10-22 15:28:57.720+00:00,47.766833,-113.753667,3.91,0.56,ml,"19 km SSE of Swan Lake, Montana",reviewed,0,5,mb,7.0,0.19900,0.11,148.0,NaN,NaN,NaN,NaN,NaN,",origin,phase-data,",",mb80527774,",",mb,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventpage/mb80527774,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=mb80527774&format=geojson,NaN,NaN,False,False,False,False,False,False,False,False,2021,10,22,Friday,13,2021-10-22,Shallow (<70 km),Below 2.0,"SSE of Swan Lake, Montana"
3,ak021dk3jgyc,2021-10-22 13:44:11.726+00:00,2021-12-06 18:54:05.882+00:00,59.678800,-136.572300,24.00,1.20,ml,"31 km WNW of Mosquito Lake, Alaska",reviewed,0,22,ak,NaN,NaN,0.26,NaN,NaN,NaN,NaN,NaN,NaN,",origin,phase-data,",",ak021dk3jgyc,",",ak,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventpage/ak021dk3jgyc,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=ak021dk3jgyc&format=geojson,NaN,NaN,False,False,False,False,False,False,False,False,2021,10,22,Friday,13,2021-10-22,Shallow (<70 km),Below 2.0,"WNW of Mosquito Lake, Alaska"
4,ci39840223,2021-10-22 13:55:21.010+00:00,2021-10-22 17:59:28.696+00:00,33.344000,-116.928500,18.04,0.79,ml,"6km WSW of Palomar Observatory, CA",reviewed,0,10,ci,25.0,0.05595,0.15,48.0,NaN,NaN,NaN,NaN,NaN,",nearby-cities,origin,phase-data,scitech-link,",",ci39840223,",",ci,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventpage/ci39840223,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=ci39840223&format=geojson,NaN,NaN,False,False,False,False,False,False,False,False,2021,10,22,Friday,13,2021-10-22,Shallow (<70 km),Below 2.0,"WSW of Palomar Observatory, CA"
